In [ ]:
import pandas as pd
import numpy as np
import json
import os
import glob
import ast
from tqdm import tqdm

In [ ]:
def generate_keys(df):
    keys = []
    for i in range(df.shape[0]):
        key = df.iloc[i, 0] + '#' + df.iloc[i, 1]
        keys.append(key)
    return keys

In [ ]:
def sorting_tablenames(df):
    sorted_df = pd.DataFrame()
    for i in range(df.shape[0]):
        ds1 = df.iloc[i,0]
        ds2 = df.iloc[i,1] 
    
        ds_esq, ds_dir = '', ''
        
        if(ds1 < ds2):
            ds_esq, ds_dir = ds1, ds2
        else:
            ds_esq, ds_dir = ds2, ds1
        
        if(len(sorted_df) == 0):
            sorted_df = pd.DataFrame({'ds_name': [ds_esq+''], 'ds_name_2': [ds_dir]})
        else:
            sorted_df = pd.concat([sorted_df, pd.DataFrame({'ds_name': [ds_esq], 'ds_name_2': [ds_dir]})], axis=0)

    sorted_df = sorted_df.drop_duplicates()
    sorted_df = sorted_df.reset_index(drop=True)

    return sorted_df

In [ ]:
def sorting_tablenames_predicted(df):
    sorted_df = pd.DataFrame()
    for i in tqdm(range(df.shape[0])):
        ds1 = df.iloc[i,0]
        ds2 = df.iloc[i,1] 
        left_column = df.iloc[i,2]
        right_column = df.iloc[i,3]
    
        ds_esq, ds_dir = '', ''
        
        if(ds1 < ds2):
            ds_esq, ds_dir, lc, rc = ds1, ds2, left_column, right_column
        else:
            ds_esq, ds_dir, lc, rc = ds2, ds1, right_column, left_column
        
        if(len(sorted_df) == 0):
            sorted_df = pd.DataFrame({'ds_name': [ds_esq+''], 'ds_name_2': [ds_dir], 'left_column': [lc], 'right_column': [rc]})
        else:
            sorted_df = pd.concat([sorted_df, pd.DataFrame({'ds_name': [ds_esq], 'ds_name_2': [ds_dir], 'left_column': [lc], 'right_column': [rc]})], axis=0)

    sorted_df = sorted_df.reset_index(drop=True)

    return sorted_df

In [ ]:
BENCHMARK = 'WebTable' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider


In [ ]:
with open(f'{BENCHMARK}/filenames.txt', 'r') as f:
    filenames = [line.strip() for line in f.readlines()]
filenames

In [ ]:
groundTruth = pd.read_csv(f'{BENCHMARK}/groundTruth.csv')
if(BENCHMARK != 'WebTable' and BENCHMARK != 'Spider'):
    groundTruth = groundTruth[groundTruth['JOINABLE_LEVEL'] == 5] # Comment this line if you want to use the Original Ground Truth

groundTruth = groundTruth[groundTruth['LEFT_TABLE'].isin(filenames)]
groundTruth = groundTruth[groundTruth['RIGHT_TABLE'].isin(filenames)]

groundTruth = groundTruth[['LEFT_TABLE', 'RIGHT_TABLE']]
groundTruth = sorting_tablenames(groundTruth)
print(groundTruth.shape)
print(groundTruth.head())

In [ ]:
jd_df = pd.read_csv(f'{BENCHMARK}/output/gpt-5.6-luna_results_processed.csv')
jd_df = sorting_tablenames_predicted(jd_df)
print(jd_df.shape)
jd_df.head()


In [ ]:
columns_descriptions_cta = pd.read_csv(f'{BENCHMARK}/column_descriptions_with_cta.csv')
columns_descriptions_cta.head()

In [ ]:
# Original Ground Truth
# remove_ctas = ['URL','Number', 'Date', 'Coordinates'] # NextiaJD-S/NextiaJD-M
# remove_ctas = ['City', 'weight', 'Gender', 'SportsTeam', 'Age', 'Time'] # Spider
# remove_ctas = ['URL', 'Coordinates', 'Gender', 'Event', 'Text', 'Time', 'Age', 'Boolean', 'Date', 'Country'] # WebTable
# remove_ctas = ['price', 'Coordinates', 'Gender', 'City', 'Product', 'Event', 'percentage', 'Age', 'Boolean', 'currency', 'Date'] # OpenData

# Revised Ground Truth
# remove_ctas = ['URL','Number', 'Date', 'Coordinates', 'category', 'Gender', 'Age'] # NextiaJD-S
# remove_ctas = ['URL','Number', 'Date', 'Coordinates','Language','Description', 'Boolean','price','CreativeWork'] # NextiaJD-M
remove_ctas = ['URL', 'Coordinates', 'Gender', 'Event', 'Text', 'Time', 'Age', 'Boolean', 'Date', 'Country'] # WebTable
# remove_ctas = ['price', 'Coordinates', 'Gender', 'City', 'Product', 'Event', 'percentage', 'Age', 'Boolean', 'currency'] # OpenData

In [ ]:
merged_cta = pd.merge(jd_df, columns_descriptions_cta, left_on=['ds_name','left_column'], right_on=['TableName', "Column"], how='inner').drop(['TableName',"Column",'Description'],axis=1)
merged_cta.columns = ['ds_name', 'ds_name_2', 'left_column', 'right_column', 'cta_left']
merged_cta = pd.merge(merged_cta, columns_descriptions_cta, left_on=['ds_name_2','right_column'], right_on=['TableName', "Column"], how='inner').drop(['TableName',"Column",'Description'],axis=1)
merged_cta.columns = ['ds_name', 'ds_name_2', 'left_column', 'right_column', 'cta_left', 'cta_right']
print(merged_cta.shape)
merged_cta = merged_cta[~merged_cta['cta_left'].isin(remove_ctas) & ~merged_cta['cta_right'].isin(remove_ctas)]
print(merged_cta.shape)
jd_df = merged_cta[['ds_name', 'ds_name_2']]
jd_df = jd_df.drop_duplicates()
jd_df.head()

In [ ]:
groundTruth['KEY'] = generate_keys(groundTruth)
jd_df['KEY'] = generate_keys(jd_df)

In [ ]:
groundTruth.head()

In [ ]:
jd_df.head()

In [ ]:
tp = 0
fp = 0
fn = 0

for i in tqdm(range(len(jd_df))):
    if(jd_df.iloc[i,2] in groundTruth.iloc[:,2].values):
        tp += 1
    else:
        fp += 1
    
for i in tqdm(range(len(groundTruth))):
    if(groundTruth.iloc[i,2] not in jd_df.iloc[:,2].values):
        fn += 1
    
precision = (tp)/(tp+fp)
recall = (tp)/(tp+fn)
f1_score = 2 * (precision*recall)/(precision+recall)

print(f"Precisão: {round(precision*100, 2)}%\nRecall: {round(recall*100, 2)}%\nF1-Score: {round(f1_score*100, 2)}%")

In [ ]:
tp, fp, fn

In [ ]:
# Printing False Positives
for i in range(len(jd_df)):
    if(jd_df.iloc[i,2] not in groundTruth.iloc[:,2].values):
        print(jd_df.iloc[i,2])

In [ ]:
# Printing False Negatives
for i in range(len(groundTruth)):
    if(groundTruth.iloc[i,2] not in jd_df.iloc[:,2].values):
        print(groundTruth.iloc[i,2])